In [29]:
# --- CELL A: GUI LAYOUT WITH DETAILED COMPENSATION TOGGLES ---
import ipywidgets as widgets
import pandas as pd

lbl_style = widgets.Layout(width='125px')
DEDUCTION_DEFAULTS = {'Single': 16100.00, 'Married Filing Jointly': 32200.00}

status_dropdown = widgets.Dropdown(options=['Single', 'Married Filing Jointly'], value='Single', description='Filing Status:')
apply_deduction_check = widgets.Checkbox(value=False, description='Apply Standard Deduction Reduction')
date_picker = widgets.DatePicker(value=pd.to_datetime('2026-06-15').date(), description='Pay Date:')

# Pair 1: Annual Base Salary
salary_lbl = widgets.Label('Annual Base:', layout=lbl_style)
salary_slide = widgets.IntSlider(value=60000, min=30000, max=250000, step=1000, continuous_update=False)
salary_box = widgets.IntText(value=60000, layout=widgets.Layout(width='90px'))
widgets.jslink((salary_slide, 'value'), (salary_box, 'value'))
salary_ui = widgets.HBox([salary_lbl, salary_slide, salary_box])

# Pair 2: Pay Periods
periods_lbl = widgets.Label('Pay Periods:', layout=lbl_style)
periods_slide = widgets.IntSlider(value=12, min=12, max=52, step=1, continuous_update=False)
periods_box = widgets.IntText(value=12, layout=widgets.Layout(width='90px'))
widgets.jslink((periods_slide, 'value'), (periods_box, 'value'))
periods_ui = widgets.HBox([periods_lbl, periods_slide, periods_box])

# Pair 3: S-Corp HIP + Payment Type Dropdown
hip_lbl = widgets.Label('S-Corp HIP:', layout=lbl_style)
hip_slide = widgets.FloatSlider(value=1310.0, min=0.0, max=2000.0, step=10.0, continuous_update=False)
hip_box = widgets.FloatText(value=1310.0, layout=widgets.Layout(width='90px'))
widgets.jslink((hip_slide, 'value'), (hip_box, 'value'))
hip_toggle = widgets.Dropdown(options=['Direct Paid by ER', 'Reimbursement to EE'], value='Direct Paid by ER', layout=widgets.Layout(width='150px'))
hip_ui = widgets.HBox([hip_lbl, hip_slide, hip_box, hip_toggle])

# Pair 4: S-Corp DIP + Payment Type Dropdown
dip_lbl = widgets.Label('S-Corp DIP:', layout=lbl_style)
dip_slide = widgets.FloatSlider(value=0.0, min=0.0, max=1000.0, step=5.0, continuous_update=False)
dip_box = widgets.FloatText(value=0.0, layout=widgets.Layout(width='90px'))
widgets.jslink((dip_slide, 'value'), (dip_box, 'value'))
dip_toggle = widgets.Dropdown(options=['Direct Paid by ER', 'Reimbursement to EE'], value='Direct Paid by ER', layout=widgets.Layout(width='150px'))
dip_ui = widgets.HBox([dip_lbl, dip_slide, dip_box, dip_toggle])

# Pair 5: 401k Deferral
retire_lbl = widgets.Label('401k Def %:', layout=lbl_style)
retire_slide = widgets.FloatSlider(value=0.0, min=0.0, max=25.0, step=0.5, continuous_update=False)
retire_box = widgets.FloatText(value=0.0, layout=widgets.Layout(width='90px'))
widgets.jslink((retire_slide, 'value'), (retire_box, 'value'))
retire_ui = widgets.HBox([retire_lbl, retire_slide, retire_box])

# Pair 6: Standard Deduction Override
deduct_lbl = widgets.Label('Std Deduction:', layout=lbl_style)
deduct_slide = widgets.FloatSlider(value=16100.0, min=0.0, max=45000.0, step=100.0, continuous_update=False)
deduct_box = widgets.FloatText(value=16100.0, layout=widgets.Layout(width='90px'))

def sync_deduct_slider_to_box(change): deduct_box.value = change['new']
def sync_deduct_box_to_slider(change):
    val = change['new']
    if val is None or str(val).strip() == "": deduct_slide.value = 0.0
    else:
        if val > deduct_slide.max: deduct_slide.max = val * 1.2
        deduct_slide.value = val

deduct_slide.observe(sync_deduct_slider_to_box, names='value')
deduct_box.observe(sync_deduct_box_to_slider, names='value')
deduct_ui = widgets.HBox([deduct_lbl, deduct_slide, deduct_box])

def on_status_changed(change):
    if deduct_slide.value in [None, 0.0, 16100.0, 32200.0]:
        deduct_slide.value = DEDUCTION_DEFAULTS[change['new']]

status_dropdown.observe(on_status_changed, 'value')
save_button = widgets.Button(description="Commit Pay Run", button_style='success', icon='check')
output_panel = widgets.Output()

input_form = widgets.VBox([status_dropdown, apply_deduction_check, deduct_ui, salary_ui, periods_ui, hip_ui, dip_ui, retire_ui, date_picker, save_button])
display(widgets.HBox([input_form, output_panel]))


In [ ]:
# --- CELL B: COMPUTATION & COMPENSATORY CASH FLOW ENGINE ---
import sqlite3
import matplotlib.pyplot as plt

DB_NAME, CSV_BACKUP_NAME, EMPLOYEE_ID = "payroll_ledger.db", "payroll_history_backup.csv", "EE-SCORP-01"
SS_WAGE_BASE, FUTA_WAGE_BASE, NM_SUI_WAGE_BASE = 184500.00, 7000.00, 34800.00   
SS_RATE, MEDICARE_RATE, FUTA_NET_RATE, NM_SUI_RATE = 0.062, 0.0145, 0.006, 0.034

TAX_TABLES_2026 = {
    'Single': {
        'FED': [(12400, 0.10), (50400, 0.12), (105700, 0.22), (201775, 0.24), (256225, 0.32), (640600, 0.35), (float('inf'), 0.37)],
        'NM':  [(8050, 0.00), (13550, 0.015), (20550, 0.032), (24550, 0.032), (33000, 0.043), (41000, 0.043), (58000, 0.047), (74000, 0.047), (102100, 0.047), (116100, 0.049), (217500, 0.049), (331100, 0.049), (float('inf'), 0.059)]
    },
    'Married Filing Jointly': {
        'FED': [(24800, 0.10), (100800, 0.12), (211400, 0.22), (403550, 0.24), (512450, 0.32), (768700, 0.35), (float('inf'), 0.37)],
        'NM':  [(16100, 0.00), (27100, 0.015), (41100, 0.032), (49100, 0.032), (66000, 0.043), (82000, 0.043), (116000, 0.047), (148000, 0.047), (204200, 0.047), (232200, 0.049), (435000, 0.049), (662200, 0.049), (float('inf'), 0.059)]
    }
}

def calc_capped(gross, ytd, cap, rate):
    return 0.0 if ytd >= cap else min(gross, cap - ytd) * rate

def calc_progressive_adjusted(gross, periods, brackets, apply_deduction, deduction_val):
    annual_taxable = max(0.00, (gross * periods) - deduction_val) if apply_deduction else (gross * periods)
    tax_owed = 0.0; prev = 0.0
    for limit, rate in brackets:
        if annual_taxable > limit:
            tax_owed += (limit - prev) * rate; prev = limit
        else:
            tax_owed += (annual_taxable - prev) * rate; break
    return tax_owed / periods

def update_dashboard(*args):
    with output_panel:
        clear_output(wait=True)
        conn = sqlite3.connect(DB_NAME); cursor = conn.cursor()
        cursor.execute("SELECT SUM(gross_wages) FROM payroll_ledger WHERE employee_id = ?", (EMPLOYEE_ID,))
        res = cursor.fetchone(); ytd_prior = float(res[0]) if res and res[0] is not None else 0.00; conn.close()
        
        status, periods = status_dropdown.value, periods_slide.value
        base_gross = salary_slide.value / periods
        hip, dip = hip_slide.value, dip_slide.value
        total_payout = base_gross + hip + dip
        deduction_401k = base_gross * (retire_slide.value / 100.0)
        
        # Calculate employee-side statutory withholding amounts
        ee_ss = calc_capped(base_gross, ytd_prior, SS_WAGE_BASE, SS_RATE)
        ee_med = base_gross * MEDICARE_RATE
        combined_tax_gross = base_gross + hip + dip - deduction_401k
        ee_fed = calc_progressive_adjusted(combined_tax_gross, periods, TAX_TABLES_2026[status]['FED'], apply_deduction_check.value, deduct_slide.value)
        ee_nm = calc_progressive_adjusted(combined_income_tax_gross:=combined_tax_gross, periods, TAX_TABLES_2026[status]['NM'], apply_deduction_check.value, deduct_slide.value)
        
        nm_wc_fee = (2.55 * 4) / periods
        total_ee_taxes = ee_ss + ee_med + ee_fed + ee_nm + nm_wc_fee
        total_net_value_package = total_payout - deduction_401k - total_ee_taxes
        
        # --- SEGMENTED CASH FLOW DISTRIBUTION RULES ---
        # 1. Labor Wages Paycheck: Base wages left over after taxes and 401k are removed
        pure_labor_check = base_gross - deduction_401k - total_ee_taxes
        
        # 2. Direct Paid by ER: Sum of policies flagged as direct corporate payments
        direct_paid_insurance = 0.0
        if hip_toggle.value == 'Direct Paid by ER': direct_paid_insurance += hip
        if dip_toggle.value == 'Direct Paid by ER': direct_paid_insurance += dip
            
        # 3. Reimbursement to EE: Sum of policies flagged as an out-of-pocket reimbursement
        reimbursement_cash = 0.0
        if hip_toggle.value == 'Reimbursement to EE': reimbursement_cash += hip
        if dip_toggle.value == 'Reimbursement to EE': reimbursement_cash += dip
            
        # Complete actual physical cash on the check received by the owner
        actual_paycheck_cash_received = pure_labor_check + reimbursement_cash
        
        # --- EMPLOYER TAXES ---
        er_ss, er_med = ee_ss, ee_med
        er_futa = calc_capped(base_gross, ytd_prior, FUTA_WAGE_BASE, FUTA_NET_RATE)
        er_suta = calc_capped(base_gross, ytd_prior, NM_SUI_WAGE_BASE, NM_SUI_RATE)
        total_er_taxes = er_ss + er_med + er_futa + er_suta + nm_wc_fee
        
        # Visual Canvas
        sizes = [max(0, pure_labor_check), deduction_401k, ee_fed, ee_nm, (ee_ss + ee_med), direct_paid_insurance, reimbursement_cash]
        names = ['Labor Check Base', '401k Def', 'Fed Tax', 'NM Tax', 'FICA taxes', 'ER Direct Insurance', 'EE Cash Reimburse']
        colors = ['#2ecc71', '#3498db', '#e74c3c', '#f1c40f', '#95a5a6', '#9b59b6', '#e67e22']
        
        combined_labels = [f"{n}\n(${v:,.2f})" for n, v in zip(names, sizes)]
        labels = [l for i, l in enumerate(combined_labels) if sizes[i] > 0]
        colors = [c for i, c in enumerate(colors) if sizes[i] > 0]
        sizes = [s for s in sizes if s > 0]
        
        fig, ax = plt.subplots(figsize=(8.5, 4.0))
        ax.pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%', startangle=140, pctdistance=0.65, labeldistance=1.15, textprops={'fontsize': 9, 'weight': 'bold'})
        ax.set_title(f"S-Corp Complete Package Allocation (Total Structural Value: ${total_net_value_package:,.2f})", fontsize=11, weight='bold', pad=20)
        plt.tight_layout(); plt.show()
        
        # Itemized Summary Output
        print("=" * 65)
        print(f"📦 TOTAL SHAREHOLDER COMPENSATION VALUE PACKAGE: ${total_net_value_package:10,.2f}")
        print("-" * 65)
        print(f"   [1] Pure Labor Wages Check Base:      ${pure_labor_check:10,.2f}")
        print(f"   [2] Employer Direct-Paid Insurance:   ${direct_paid_insurance:10,.2f}")
        print(f"   [3] Tax-Free Out-of-Pocket Reimburse: ${reimbursement_cash:10,.2f}")
        print("-" * 65)
        print(f"💵 PHYSICAL NET CASH ON PAYCHECK DISTRIBUTION:   ${actual_paycheck_cash_received:10,.2f}")
        print(f"🏢 TOTAL CORPORATE TRANSACTION OUTFLOW COST:    ${(total_payout + total_er_taxes):10,.2f}")
        print("=" * 65)

# Bind all active toggles to refresh screen layout instantly
for w in [status_dropdown, apply_deduction_check, salary_slide, periods_slide, hip_slide, dip_slide, retire_slide, deduct_slide, date_picker, hip_toggle, dip_toggle]:
    w.observe(update_dashboard, 'value')
update_dashboard()
